# 03b · Kontrol deneyi: telafi korunur, ana kola akım da verilir (5 seed)
**Soru:** −20 °C kazancı *telafiden* mi geliyor, yoksa *akımın ana koldan çıkarılmasından* mı?
Hibrit'te iki şey birden farklı (akım var, telafi yok). Burada telafi aynen kalır, yalnız ana kola akım eklenir:

| Varyant | Ana kol girdisi | Fizik dalı |
|---|---|---|
| Önerilen (02, final) | V, T, V̄, V − η̂ | ohmik + 3 RC |
| **RC + akım girişi** | **V, I, T, V̄, V − η̂** | ohmik + 3 RC |
| RC: yalnız ohmik kol (03 + 2 ek seed) | V, T, V̄, V − η̂ | yalnız ohmik |
| **RC: yalnız ohmik + akım girişi** | **V, I, T, V̄, V − η̂** | yalnız ohmik |
| Hibrit (02, final) | V, I, T, V̄ | yok |

Seed'ler 101/202/303/404/505 (final ile aynı → seed×çevrim eşleştirilmiş test). Hiperparametreler seçilen önerilen modelinkiyle aynı (yeniden ayar yok).
Yalnız-ohmik varyant 03'te 3 seed'di; 404/505 eklenerek 5'e tamamlanır (aynı `ablation` klasörüne yazılır).
Kesilirse yeniden çalıştırın: tamamlanan koşular atlanır. Tahmini süre: koşu başına ~45–60 dk → 12 koşu ≈ 9–12 saat.

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
sel = exp.load_selected()
cur = current_input_specs(sel)
abl, _ = ablation_specs(sel)
SEEDS = exp.cfg['FINAL_SEEDS']
t0 = time.time()
plan = [(n, s, sd, 'ablation_b') for n, s in cur.items() for sd in SEEDS] + \
       [('RC: yalnız ohmik kol', abl['RC: yalnız ohmik kol'], sd, 'ablation') for sd in SEEDS]
for i, (name, spec, seed, stage) in enumerate(plan, 1):
    exp.run(name, spec, seed, stage, ablations=RC_ABLATIONS_TEST)
    print(f'[{i}/{len(plan)}] {name} s{seed} · geçen {(time.time() - t0) / 3600:.2f} sa')

In [ ]:
from scipy import stats
from models.report import rc_param_table
OURS, HYB, OHM = 'Önerilen (RC-Hibrit)', 'Hibrit (GRU→Transformer)', 'RC: yalnız ohmik kol'
fin = exp.collect('final'); ab = exp.collect('ablation'); b = exp.collect('ablation_b')
D = pd.concat([fin[fin.model.isin([OURS, HYB])], ab[ab.model == OHM], b], ignore_index=True)
D = D[D.seed.isin(SEEDS)]
D['k'] = np.where(D.temp == '-20degC', '−20°C', 'eğitim sıc.')
ORDER = [OURS, 'RC + akım girişi', OHM, 'RC: yalnız ohmik + akım girişi', HYB]

sl = D.groupby(['model', 'k', 'seed'])[['rmse', 'bias', 'jitter']].mean() * 100
ozet = sl.groupby(['model', 'k']).agg(['mean', 'std'])
ozet.columns = [f'{a}_{b}' for a, b in ozet.columns]
ozet['n_seed'] = sl.groupby(['model', 'k']).size()
ozet = ozet.reset_index(); ozet['model'] = pd.Categorical(ozet.model, ORDER); ozet = ozet.sort_values(['k', 'model'])
print(ozet.round(3).to_string(index=False))

In [ ]:
def pair(a, bm, k):
    """a vs bm: seed düzeyi Welch + eşleştirilmiş Wilcoxon (−20: seed×çevrim n=20; eğitim sıc.: seed×sıcaklık×çevrim n=80)."""
    m = D[D.k == k]
    s = m.groupby(['model', 'seed']).rmse.mean() * 100
    c = m.groupby(['model', 'seed', 'temp', 'cycle']).rmse.mean() * 100
    x = c.loc[a]; y = c.loc[bm].reindex(x.index); d = (x - y).dropna()
    return {'karşılaştırma': f'{a}  vs  {bm}', 'kapsam': k, 'Δ RMSE (puan, a−b)': s.loc[a].mean() - s.loc[bm].mean(),
            'Welch p (seed)': stats.ttest_ind(s.loc[a], s.loc[bm], equal_var=False).pvalue,
            'a daha iyi': f'{(d < 0).sum()}/{len(d)}', 'Wilcoxon p (eşleştirilmiş)': stats.wilcoxon(d).pvalue if (d != 0).any() else np.nan}
PAIRS = [(OURS, 'RC + akım girişi'),                    # ana soru: akım eklemek −20'yi bozuyor mu?
         (OHM, 'RC: yalnız ohmik + akım girişi'),        # aynı soru, yalnız ohmik dalda
         ('RC + akım girişi', HYB),                      # akım varken telafinin katkısı
         (OURS, OHM)]                                    # RC kolları: 5 seed ile yeniden
sig = pd.DataFrame([pair(a, bm, k) for a, bm in PAIRS for k in ['−20°C', 'eğitim sıc.']])
print(sig.round(4).to_string(index=False))

In [ ]:
# Öğrenilen fizik parametreleri: akım ana kola verilince η dalı fiziksel kalıyor mu?
recs = [r for r in exp.records('final') if r['name'] == OURS] + exp.records('ablation_b') + \
       [r for r in exp.records('ablation') if r['name'] == OHM]
rcp = rc_param_table(recs, exp.norm)
cols = ['model', 'seed'] + [c for c in rcp.columns if c.startswith('R0') or c.startswith('ΣRk') or c.startswith('τ')]
print(rcp[cols].groupby('model').mean(numeric_only=True).drop(columns='seed').round(1).T.to_string())

In [ ]:
with pd.ExcelWriter(exp.p['excel'] / '03b_akim_girisi.xlsx') as xw:
    ozet.to_excel(xw, sheet_name='Özet', index=False)
    sig.to_excel(xw, sheet_name='Anlamlılık', index=False)
    rcp[cols].to_excel(xw, sheet_name='RC parametreleri', index=False)
    D.to_excel(xw, sheet_name='Ham satırlar', index=False)
print('✅', exp.p['excel'] / '03b_akim_girisi.xlsx')

### Nasıl okunur
- **RC + akım girişi −20 °C'de belirgin kötüleşir** (Önerilen vs RC + akım: Δ < 0, p < 0.05) → akımsız omurga *gerekli*; ağ akımı görünce sıcaklığa özgü akım–gerilim kısayolunu yeniden öğreniyor. Ana iddia nedensel olarak güçlenir.
- **Fark yoksa** → kazanç telafinin kendisinden; akımı çıkarmak zararsız bir sadeleştirme. Makalede böyle yazılır (yine geçerli katkı).
- **RC + akım girişi vs Hibrit:** akım varken bile telafinin −20 °C katkısını gösterir.
- RC parametreleri tablosu: akım verilince R0/τ değerleri fiziksel aralıktan kayıyor mu (yorumlanabilirlik)?
- Önerilen vs yalnız ohmik: artık 5 seed; RC kolları hakkındaki ifadeyi buna göre kesinleştiririz.